<img src="http://developer.download.nvidia.com/compute/machine-learning/frameworks/nvidia_logo.png" align="right" width="100px"/>

# LangGraph Agent for Parse + Omni OCR Word Histograms

This notebook builds a Colab-ready LangGraph agent that counts non-stop words found in OCR text from document pages.

The pipeline uses three separate roles:

| Role | Implementation | Why it exists |
| --- | --- | --- |
| OCR/layout stage | **Nemotron Parse 1.2** endpoint, `nvidia/nemotron-parse` | Reads page layout and returns OCR text blocks in page order. |
| Visual OCR helper + agent model | **Nemotron 3 Nano Omni**, `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` | OCRs text inside Parse `Picture` crops and decides which tool action to run. |
| Deterministic tool | Python function wrapped as a LangGraph tool | Counts lowercased words exactly after English stop-word and numeric-only token removal and can be unit-tested without model calls. |

You do not need a GPU. Model calls go to NVIDIA's hosted endpoint at `https://integrate.api.nvidia.com/v1` using a Colab Secret named `NVIDIA_API_KEY`.


## 1. Setup

Run this in Google Colab. The deterministic unit tests later in the notebook do not call any API, but the full smoke test requires `NVIDIA_API_KEY` in Colab Secrets.


In [1]:
!uv pip install --quiet langgraph==1.2.1 langchain-core==1.4.0 pymupdf==1.27.2.3
print("[setup] OK -- runtime dependencies installed.")


[setup] OK -- runtime dependencies installed.


## 2. API key and shared imports

The notebook first tries Colab Secrets. If you run locally, it falls back to an environment variable with the same name.


In [2]:
from __future__ import annotations

import base64
import io
import json
import os
import re
import textwrap
import time
from collections import Counter
from pathlib import Path
from typing import Any, Literal

import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import Markdown, display
from PIL import Image

try:
    from google.colab import userdata  # type: ignore
    try:
        secret = userdata.get("NVIDIA_API_KEY")
        if secret:
            os.environ["NVIDIA_API_KEY"] = secret
            print("[auth] NVIDIA_API_KEY loaded from Colab Secrets.")
    except Exception as exc:
        print("[auth] Colab Secret unavailable or notebook access is disabled:", exc)
except Exception:
    print("[auth] Not running in Colab; using local environment variables.")

NOTEBOOK_ROOT = Path.cwd()
API_KEY = os.environ.get("NVIDIA_API_KEY", "")
NVAI_URL = os.environ.get("NVAI_URL", "https://integrate.api.nvidia.com/v1")
PARSE_MODEL = "nvidia/nemotron-parse"  # Hosted Nemotron Parse 1.2 endpoint
NANO_OMNI_MODEL = os.environ.get(
    "NANO_OMNI_MODEL",
    "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning",
)

print(f"Endpoint:             {NVAI_URL}")
print(f"OCR/layout model:     {PARSE_MODEL}  (Nemotron Parse 1.2)")
print(f"Agent/OCR helper:     {NANO_OMNI_MODEL}")
if not API_KEY:
    print("[auth] NVIDIA_API_KEY is not set yet. The unit tests can still run; the API smoke test will skip.")


[auth] NVIDIA_API_KEY loaded from Colab Secrets.
Endpoint:             https://integrate.api.nvidia.com/v1
OCR/layout model:     nvidia/nemotron-parse  (Nemotron Parse 1.2)
Agent/OCR helper:     nvidia/nemotron-3-nano-omni-30b-a3b-reasoning


## 3. Demo inputs

`PAGE_INPUTS` is the only input list you need to edit. Use `kind="pdf"` with a one-indexed `page_number`, or use `kind="image"` for image files.


In [3]:
DOC_DIR = NOTEBOOK_ROOT / "data" / "documents"
DOC_DIR.mkdir(parents=True, exist_ok=True)

_HF_DOC_ROOT = (
    "https://huggingface.co/datasets/yubo2333/MMLongBench-Doc/"
    "resolve/main/documents"
)

PAGE_INPUTS: list[dict[str, Any]] = [
    {
        "source_id": "pew-page-5",
        "kind": "pdf",
        "path": str(DOC_DIR / "05-03-18-political-release.pdf"),
        "page_number": 5,
        "url": f"{_HF_DOC_ROOT}/05-03-18-political-release.pdf",
    }
]


def ensure_input_file(spec: dict[str, Any]) -> Path:
    """Return a local input path, downloading demo PDFs only when needed."""
    path = Path(spec["path"])
    if path.exists():
        return path
    url = spec.get("url")
    if not url:
        raise FileNotFoundError(
            f"Missing input file {path}. Add the file or provide a 'url' in PAGE_INPUTS."
        )
    print(f"[download] {path.name} <- {url}")
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(response.content)
    return path


def pdf_page_to_image(pdf_path: str | Path, page_number: int, *, dpi: int = 150) -> Image.Image:
    """Render a one-indexed PDF page to an RGB PIL image."""
    doc = fitz.open(pdf_path)
    try:
        page = doc.load_page(page_number - 1)
        zoom = dpi / 72.0
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    finally:
        doc.close()


def load_page_image(spec: dict[str, Any], *, dpi: int = 150) -> Image.Image:
    path = ensure_input_file(spec)
    kind = spec.get("kind", "pdf")
    if kind == "pdf":
        return pdf_page_to_image(path, int(spec.get("page_number", 1)), dpi=dpi)
    if kind == "image":
        return Image.open(path).convert("RGB")
    raise ValueError(f"Unsupported PAGE_INPUTS kind: {kind!r}")


## 4. Pure Python tool logic

These functions do the exact word counting and OCR-text assembly. They are independent of NVIDIA APIs, LangGraph, and Colab, which is why we can test them directly.


In [ ]:
# PURE_PYTHON_CELL
from __future__ import annotations
from collections import Counter
from typing import Any
import re

TEXT_BLOCK_TYPES = {
    "Title",
    "Section-header",
    "Text",
    "List-item",
    "Caption",
    "Table",
    "Formula",
    "Page-header",
    "Page-footer",
    "Footnote",
    "Bibliography",
    "TOC",
}
PICTURE_BLOCK_TYPES = {"Picture", "Figure"}

WORD_PATTERN = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*")
ENGLISH_STOP_WORDS = frozenset(
    {
        "a", "about", "above", "after", "again", "against", "all", "am", "an", "and",
        "any", "are", "as", "at", "be", "because", "been", "before", "being", "below",
        "between", "both", "but", "by", "can", "could", "did", "do", "does", "doing",
        "down", "during", "each", "few", "for", "from", "further", "had", "has", "have",
        "having", "he", "her", "here", "hers", "herself", "him", "himself", "his", "how",
        "i", "if", "in", "into", "is", "it", "its", "itself", "just", "me", "more",
        "most", "my", "myself", "no", "nor", "not", "now", "of", "off", "on", "once",
        "only", "or", "other", "our", "ours", "ourselves", "out", "over", "own", "same",
        "she", "should", "so", "some", "such", "than", "that", "the", "their", "theirs",
        "them", "themselves", "then", "there", "these", "they", "this", "those", "through",
        "to", "too", "under", "until", "up", "very", "was", "we", "were", "what", "when",
        "where", "which", "while", "who", "whom", "why", "will", "with", "would", "you",
        "your", "yours", "yourself", "yourselves",
    }
)


def normalize_word(raw_word: str) -> str:
    """Lowercase a token and remove a simple trailing possessive."""
    word = raw_word.lower()
    if word.endswith("'s") and len(word) > 2:
        word = word[:-2]
    return word


def has_alphabetic_character(word: str) -> bool:
    return any(char.isalpha() for char in word)


def iter_countable_words(text: str):
    """Yield lowercased words that are not stop words or numeric-only tokens."""
    normalized_text = text.replace("\u2019", "'")
    for match in WORD_PATTERN.finditer(normalized_text):
        word = normalize_word(match.group(0))
        if word and has_alphabetic_character(word) and word not in ENGLISH_STOP_WORDS:
            yield word


def top_word_rows(counts: dict[str, int], *, top_n: int = 20) -> list[dict[str, Any]]:
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:top_n]
    return [{"word": word, "count": count} for word, count in ordered]


def count_words(text: str) -> dict[str, int]:
    """Count lowercased non-stop words in text."""
    counts: Counter[str] = Counter(iter_countable_words(text))
    return dict(sorted(counts.items(), key=lambda item: item[0]))


def make_word_histograms(pages: list[dict[str, Any]]) -> dict[str, Any]:
    """Build per-page and aggregate word histograms from OCR text.

    Args:
        pages: A list of dictionaries with `page_number` and `text` keys.

    Returns:
        A JSON-serializable dictionary with exact counts and top-word summaries.
    """
    page_results: list[dict[str, Any]] = []
    aggregate: Counter[str] = Counter()

    for index, page in enumerate(pages, start=1):
        page_number = int(page.get("page_number", index))
        text = str(page.get("text", ""))
        counts = count_words(text)
        aggregate.update(counts)
        counted_total = sum(counts.values())
        page_results.append(
            {
                "page_number": page_number,
                "text_length": len(text),
                "counted_words": counted_total,
                "unique_words": len(counts),
                "counts": counts,
                "top_words": top_word_rows(counts),
            }
        )

    aggregate_counts = dict(sorted(aggregate.items(), key=lambda item: item[0]))
    return {
        "settings": {
            "lowercase": True,
            "stop_words_removed": True,
            "numeric_only_tokens_removed": True,
            "stop_word_language": "english",
            "word_pattern": WORD_PATTERN.pattern,
        },
        "page_count": len(page_results),
        "total_counted_words": sum(aggregate_counts.values()),
        "unique_words": len(aggregate_counts),
        "pages": page_results,
        "aggregate_counts": aggregate_counts,
        "top_words": top_word_rows(aggregate_counts),
    }


def block_text(block: dict[str, Any]) -> str:
    """Extract text from common Nemotron Parse block shapes."""
    for key in ("text", "markdown", "content"):
        value = block.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""


def assemble_ocr_text_from_blocks(
    parse_blocks: list[dict[str, Any]],
    picture_texts: list[str] | None = None,
) -> str:
    """Weave Parse OCR text and picture OCR text in Parse reading order."""
    parts: list[str] = []
    picture_iter = iter(picture_texts or [])

    for block in parse_blocks:
        block_type = block.get("type")
        if block_type in PICTURE_BLOCK_TYPES:
            picture_text = next(picture_iter, "").strip()
            if picture_text:
                parts.append(picture_text)
            continue
        if block_type in TEXT_BLOCK_TYPES or block.get("text"):
            text = block_text(block)
            if text:
                parts.append(text)

    return "\n".join(parts)


## 5. Unit tests for the deterministic tool

This cell has no network calls. It is meant to prove that the Python tool is correct independent of model behavior.


In [ ]:
# PURE_PYTHON_CELL
import unittest


class WordHistogramTests(unittest.TestCase):
    def test_counts_lowercase_words_and_removes_stop_words(self):
        result = make_word_histograms([{"page_number": 1, "text": "The GPU and the gpu!"}])
        self.assertEqual(result["pages"][0]["counts"], {"gpu": 2})
        self.assertEqual(result["total_counted_words"], 2)
        self.assertTrue(result["settings"]["lowercase"])
        self.assertTrue(result["settings"]["stop_words_removed"])

    def test_possessive_and_hyphenated_words(self):
        result = make_word_histograms(
            [{"page_number": 1, "text": "NVIDIA's GPU-accelerated tools"}]
        )
        self.assertEqual(
            result["pages"][0]["counts"],
            {"gpu-accelerated": 1, "nvidia": 1, "tools": 1},
        )

    def test_multi_page_aggregate_counts(self):
        result = make_word_histograms(
            [
                {"page_number": 1, "text": "GPU tools"},
                {"page_number": 2, "text": "gpu systems tools"},
            ]
        )
        self.assertEqual(
            result["aggregate_counts"],
            {"gpu": 2, "systems": 1, "tools": 2},
        )
        self.assertEqual(result["page_count"], 2)
        self.assertEqual(result["unique_words"], 3)

    def test_numeric_only_tokens_are_removed(self):
        result = make_word_histograms(
            [{"page_number": 1, "text": "Trump 54 43 confidence H100 3D 54's 3-4"}]
        )
        self.assertEqual(
            result["pages"][0]["counts"],
            {"3d": 1, "confidence": 1, "h100": 1, "trump": 1},
        )
        self.assertNotIn("54", result["aggregate_counts"])
        self.assertNotIn("43", result["aggregate_counts"])
        self.assertTrue(result["settings"]["numeric_only_tokens_removed"])

    def test_ocr_assembly_preserves_parse_reading_order(self):
        blocks = [
            {"type": "Text", "text": "Alpha"},
            {"type": "Picture", "bbox": {"xmin": 0, "ymin": 0, "xmax": 1, "ymax": 1}},
            {"type": "Text", "text": "Omega"},
        ]
        self.assertEqual(
            assemble_ocr_text_from_blocks(blocks, ["Beta"]),
            "Alpha\nBeta\nOmega",
        )


suite = unittest.defaultTestLoader.loadTestsFromTestCase(WordHistogramTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
if not result.wasSuccessful():
    raise AssertionError("Word histogram tests failed")


## 6. NVIDIA API surfaces

These helpers follow the same request style as the Nemotron Parse + Omni cookbook: raw HTTPS calls, data-URL images, forced Parse tool use, and `/no_think` cleanup for direct Omni calls.


In [6]:
def require_nvidia_api_key() -> str:
    key = os.environ.get("NVIDIA_API_KEY", "")
    if not key:
        raise RuntimeError(
            "NVIDIA_API_KEY is not set. In Colab, open the Secrets pane, add "
            "a secret named NVIDIA_API_KEY, and enable notebook access."
        )
    return key


def nvidia_headers() -> dict[str, str]:
    return {
        "Authorization": f"Bearer {require_nvidia_api_key()}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }


def pil_to_data_url(img: Image.Image, *, fmt: str = "JPEG", quality: int = 85) -> str:
    if img.mode != "RGB":
        img = img.convert("RGB")
    buf = io.BytesIO()
    if fmt.upper() == "JPEG":
        img.save(buf, format="JPEG", quality=quality)
        mime = "image/jpeg"
    else:
        img.save(buf, format="PNG")
        mime = "image/png"
    return f"data:{mime};base64," + base64.b64encode(buf.getvalue()).decode()


def call_nemotron_parse(image: Image.Image) -> list[dict[str, Any]]:
    """Run Nemotron Parse 1.2 on a page image and return layout/OCR blocks."""
    body = {
        "model": PARSE_MODEL,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": pil_to_data_url(image, fmt="PNG")}}
                ],
            }
        ],
        "tools": [{"type": "function", "function": {"name": "markdown_bbox"}}],
        "tool_choice": {"type": "function", "function": {"name": "markdown_bbox"}},
        "max_tokens": 8192,
        "temperature": 0.0,
    }
    response = requests.post(
        f"{NVAI_URL}/chat/completions",
        headers=nvidia_headers(),
        json=body,
        timeout=180,
    )
    response.raise_for_status()
    args = response.json()["choices"][0]["message"]["tool_calls"][0]["function"]["arguments"]
    parsed = json.loads(args)
    blocks = parsed if isinstance(parsed, list) else parsed.get("tool_call_arguments", [])
    if blocks and isinstance(blocks[0], list):
        blocks = blocks[0]
    return blocks or []


_SYS_NO_THINK = (
    "/no_think\n"
    "Answer directly and concisely. Do NOT include any reasoning, preamble, or <think> blocks."
)
_THINK_RE = re.compile(r"<think\b[^>]*>.*?</think>", re.DOTALL | re.IGNORECASE)
_SYSTEM_ECHO = re.compile(
    r"^\s*(?:/no_think\s*)?Answer directly and concisely[^.]*\.\s*"
    r"Do NOT include any reasoning[^.]*\.\s*",
    re.IGNORECASE,
)


def call_nano_omni(
    prompt: str,
    images: list[Image.Image] | None = None,
    *,
    reasoning: Literal["on", "off"] = "off",
    json_mode: bool = False,
    temperature: float = 0.2,
    top_p: float = 0.95,
    max_tokens: int = 2048,
) -> dict[str, Any]:
    parts: list[dict[str, Any]] = [{"type": "text", "text": prompt}]
    for img in images or []:
        parts.append({"type": "image_url", "image_url": {"url": pil_to_data_url(img)}})

    messages: list[dict[str, Any]] = []
    if reasoning == "off":
        messages.append({"role": "system", "content": _SYS_NO_THINK})
    messages.append({"role": "user", "content": parts})

    body: dict[str, Any] = {
        "model": NANO_OMNI_MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stream": False,
        "chat_template_kwargs": {"enable_thinking": reasoning == "on"},
    }
    if reasoning == "on":
        body["top_p"] = top_p
    else:
        body["top_k"] = 1
    if json_mode:
        body["response_format"] = {"type": "json_object"}

    response = requests.post(
        f"{NVAI_URL}/chat/completions",
        headers=nvidia_headers(),
        json=body,
        timeout=240,
    )
    response.raise_for_status()
    return response.json()


def extract_text(resp: dict[str, Any]) -> str:
    msg = resp.get("choices", [{}])[0].get("message", {})
    text = (msg.get("content") or msg.get("reasoning") or "").strip()
    text = _THINK_RE.sub("", text).strip()
    text = _SYSTEM_ECHO.sub("", text, count=1).strip()
    return text


def extract_json_object(resp: dict[str, Any]) -> dict[str, Any]:
    msg = resp.get("choices", [{}])[0].get("message", {})
    for raw in (msg.get("content") or "", msg.get("reasoning") or ""):
        s = _THINK_RE.sub("", (raw or "").strip()).strip()
        if not s:
            continue
        if s.startswith("```"):
            s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE).strip("` \n")
        try:
            obj = json.loads(s)
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            pass
        left, right = s.find("{"), s.rfind("}")
        if left != -1 and right > left:
            try:
                obj = json.loads(s[left : right + 1])
                if isinstance(obj, dict):
                    return obj
            except json.JSONDecodeError:
                pass
    return {}


def crop_block(page: Image.Image, bbox: dict[str, float]) -> Image.Image:
    width, height = page.size
    return page.crop(
        (
            int(bbox["xmin"] * width),
            int(bbox["ymin"] * height),
            int(bbox["xmax"] * width),
            int(bbox["ymax"] * height),
        )
    )


def ocr_picture_text(crop: Image.Image) -> str:
    prompt = (
        "OCR all visible text in this image crop. Preserve words, numbers, "
        "punctuation, and line breaks when you can. Return only the visible text. "
        "If there is no readable text, return an empty string."
    )
    response = call_nano_omni(
        prompt,
        images=[crop],
        reasoning="off",
        json_mode=False,
        temperature=0.0,
        max_tokens=2048,
    )
    return extract_text(response).strip()


## 7. OCR extraction stage

Parse handles page text and layout. When Parse marks a region as `Picture`, Nano Omni OCRs that crop so text inside screenshots, charts, or infographics still reaches the word histogram tool.


In [7]:
def extract_ocr_page(spec: dict[str, Any]) -> dict[str, Any]:
    page_number = int(spec.get("page_number", 1))
    source_id = str(spec.get("source_id", Path(spec.get("path", "input")).stem))
    image = load_page_image(spec)

    t0 = time.time()
    parse_blocks = call_nemotron_parse(image)
    parse_seconds = time.time() - t0

    picture_texts: list[str] = []
    for block in parse_blocks:
        if block.get("type") in PICTURE_BLOCK_TYPES and block.get("bbox"):
            crop = crop_block(image, block["bbox"])
            picture_texts.append(ocr_picture_text(crop))

    text = assemble_ocr_text_from_blocks(parse_blocks, picture_texts)
    return {
        "source_id": source_id,
        "page_number": page_number,
        "text": text,
        "parse_block_count": len(parse_blocks),
        "picture_ocr_count": len(picture_texts),
        "parse_seconds": round(parse_seconds, 2),
        "parse_blocks": parse_blocks,
    }


def extract_ocr_pages(page_inputs: list[dict[str, Any]]) -> list[dict[str, Any]]:
    pages = []
    for spec in page_inputs:
        print(f"[ocr] {spec.get('source_id', spec.get('path'))} page {spec.get('page_number', 1)}")
        page = extract_ocr_page(spec)
        print(
            f"      {len(page['text'])} OCR chars, "
            f"{page['parse_block_count']} Parse blocks, "
            f"{page['picture_ocr_count']} picture crop(s) OCRed"
        )
        pages.append(page)
    return pages


## 8. LangGraph agent with a real Python tool

Nano Omni emits a JSON tool action. The graph validates that action, injects the authoritative OCR text from state, converts it into a LangChain tool call, and executes it through `ToolNode`.


In [ ]:
from typing_extensions import Annotated, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.constants import END, START
from langgraph.graph import StateGraph, add_messages
from langgraph.prebuilt import ToolNode


@tool
def make_word_histograms_tool(pages: list[dict[str, Any]]) -> str:
    """Count lowercased non-stop words in OCR text for each page and in aggregate.

    Args:
        pages: List of objects with page_number and text keys.
    """
    return json.dumps(make_word_histograms(pages), ensure_ascii=False)


class HistogramAgentState(TypedDict, total=False):
    page_inputs: list[dict[str, Any]]
    pages: list[dict[str, Any]]
    messages: Annotated[list[AnyMessage], add_messages]
    tool_result: dict[str, Any]
    summary: str


def extract_ocr_node(state: HistogramAgentState) -> dict[str, Any]:
    pages = extract_ocr_pages(state.get("page_inputs", PAGE_INPUTS))
    page_summaries = [
        {
            "page_number": p["page_number"],
            "source_id": p["source_id"],
            "ocr_chars": len(p["text"]),
            "picture_ocr_count": p["picture_ocr_count"],
        }
        for p in pages
    ]
    return {
        "pages": pages,
        "messages": [HumanMessage(content="OCR pages extracted: " + json.dumps(page_summaries))],
    }


def build_tool_action_prompt(state: HistogramAgentState) -> str:
    page_manifest = [
        {
            "page_number": p["page_number"],
            "source_id": p.get("source_id", ""),
            "ocr_chars": len(p.get("text", "")),
            "text_preview": p.get("text", "")[:700],
        }
        for p in state.get("pages", [])
    ]
    return f"""
You are a LangGraph agent that must choose a tool action for OCR word counting.

Available tool:
- make_word_histograms_tool(pages)

Return exactly one JSON object with this shape:
{{
  "tool_name": "make_word_histograms_tool",
  "args": {{}},
  "reason": "short reason"
}}

Counting settings are fixed: words are lowercased, simple trailing possessives are stripped, common English stop words are removed, and numeric-only tokens are removed.

OCR page manifest:
{json.dumps(page_manifest, ensure_ascii=False, indent=2)}
""".strip()


def coerce_tool_action(raw_action: dict[str, Any], state: HistogramAgentState) -> dict[str, Any]:
    """Validate the model action and preserve OCR text from graph state."""
    return {
        "tool_name": "make_word_histograms_tool",
        "args": {
            "pages": [
                {"page_number": p["page_number"], "text": p.get("text", "")}
                for p in state.get("pages", [])
            ],
        },
        "reason": str(raw_action.get("reason", "Validated default word histogram action.")),
    }


def agent_node(state: HistogramAgentState) -> dict[str, Any]:
    response = call_nano_omni(
        build_tool_action_prompt(state),
        images=None,
        reasoning="off",
        json_mode=True,
        temperature=0.0,
        max_tokens=512,
    )
    raw_action = extract_json_object(response)
    action = coerce_tool_action(raw_action, state)
    tool_call = {
        "name": action["tool_name"],
        "args": action["args"],
        "id": "call_word_histograms",
        "type": "tool_call",
    }
    return {
        "messages": [AIMessage(content=json.dumps(action, ensure_ascii=False), tool_calls=[tool_call])]
    }


def latest_tool_json(messages: list[AnyMessage]) -> dict[str, Any]:
    for message in reversed(messages):
        if isinstance(message, ToolMessage):
            content = message.content
            if isinstance(content, str):
                return json.loads(content)
    raise ValueError("No ToolMessage with histogram JSON found in graph state.")


def summarize_node(state: HistogramAgentState) -> dict[str, Any]:
    tool_result = latest_tool_json(state.get("messages", []))
    compact = {
        "page_count": tool_result["page_count"],
        "total_counted_words": tool_result["total_counted_words"],
        "unique_words": tool_result["unique_words"],
        "top_words": tool_result["top_words"][:10],
        "settings": tool_result["settings"],
    }
    prompt = (
        "Summarize this OCR word histogram result in 3 concise bullets. "
        "Mention that stop words and numeric-only tokens were removed, then name the most frequent words.\n\n"
        + json.dumps(compact, ensure_ascii=False, indent=2)
    )
    response = call_nano_omni(
        prompt,
        images=None,
        reasoning="off",
        json_mode=False,
        temperature=0.2,
        max_tokens=512,
    )
    return {"tool_result": tool_result, "summary": extract_text(response)}


tool_node = ToolNode([make_word_histograms_tool])

builder = StateGraph(HistogramAgentState)
builder.add_edge(START, "extract_ocr")
builder.add_node("extract_ocr", extract_ocr_node)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_node("summarize", summarize_node)
builder.add_edge("extract_ocr", "agent")
builder.add_edge("agent", "tools")
builder.add_edge("tools", "summarize")
builder.add_edge("summarize", END)

app = builder.compile()
print("LangGraph app ready: extract_ocr -> agent -> tools -> summarize")


## 9. Plotting helpers

The bar charts show the most frequent words after lowercasing, stop-word removal, and numeric-only token removal.


In [ ]:
def counts_to_frame(counts: dict[str, int], *, top_n: int = 25) -> pd.DataFrame:
    rows = top_word_rows(counts, top_n=top_n)
    return pd.DataFrame(rows)


def plot_count_axis(ax, counts: dict[str, int], title: str, *, top_n: int = 25) -> None:
    rows = top_word_rows(counts, top_n=top_n)
    labels = [row["word"] for row in rows]
    values = [row["count"] for row in rows]
    ax.bar(labels, values, color="#76B900")
    ax.set_title(title)
    ax.set_xlabel("Word")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)


def histogram_plot_items(histogram_result: dict[str, Any]) -> list[dict[str, Any]]:
    pages = histogram_result.get("pages", [])
    items = [
        {
            "counts": page["counts"],
            "title": f"Page {page['page_number']} word histogram",
        }
        for page in pages
    ]
    if len(pages) != 1:
        items.append(
            {
                "counts": histogram_result.get("aggregate_counts", {}),
                "title": "Aggregate word histogram",
            }
        )
    return items


def plot_histograms(histogram_result: dict[str, Any], *, top_n: int = 25):
    items = histogram_plot_items(histogram_result)
    nrows = len(items)
    fig_height = max(4, 3.3 * nrows)
    fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=(12, fig_height), squeeze=False)

    for ax, item in zip(axes[:, 0], items):
        plot_count_axis(
            ax,
            item["counts"],
            item["title"],
            top_n=top_n,
        )

    fig.tight_layout()
    return fig


In [ ]:
# PURE_PYTHON_CELL
import unittest


class HistogramPlotTests(unittest.TestCase):
    def test_single_page_plot_items_skip_duplicate_aggregate(self):
        result = {
            "pages": [{"page_number": 5, "counts": {"confidence": 7, "trump": 7}}],
            "aggregate_counts": {"confidence": 7, "trump": 7},
        }
        self.assertEqual(
            [item["title"] for item in histogram_plot_items(result)],
            ["Page 5 word histogram"],
        )

    def test_multi_page_plot_items_include_aggregate(self):
        result = {
            "pages": [
                {"page_number": 1, "counts": {"alpha": 2}},
                {"page_number": 2, "counts": {"beta": 3}},
            ],
            "aggregate_counts": {"alpha": 2, "beta": 3},
        }
        self.assertEqual(
            [item["title"] for item in histogram_plot_items(result)],
            [
                "Page 1 word histogram",
                "Page 2 word histogram",
                "Aggregate word histogram",
            ],
        )


suite = unittest.defaultTestLoader.loadTestsFromTestCase(HistogramPlotTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
if not result.wasSuccessful():
    raise AssertionError("Histogram plotting tests failed")


## 10. Guarded API smoke test

This cell runs the full agent on the first demo page when `NVIDIA_API_KEY` is available. If the key is missing, it prints a skip message and leaves the unit-tested Python tool available for inspection.


In [ ]:
if not os.environ.get("NVIDIA_API_KEY"):
    print("[skip] NVIDIA_API_KEY is not set, so the Parse + Omni API smoke test was skipped.")
else:
    initial_state: HistogramAgentState = {
        "page_inputs": PAGE_INPUTS[:1],
        "messages": [],
    }
    final_state = app.invoke(initial_state)

    display(Markdown("### Agent summary"))
    display(Markdown(final_state["summary"]))

    histogram_result = final_state["tool_result"]
    display(Markdown("### Aggregate top words"))
    display(counts_to_frame(histogram_result["aggregate_counts"], top_n=15))

    fig = plot_histograms(histogram_result, top_n=25)
    display(fig)


## 11. Try your own pages

Edit `PAGE_INPUTS` near the top of the notebook. For PDFs, keep `page_number` one-indexed, matching the number shown in a PDF viewer. For images, set `kind` to `"image"` and omit `page_number` or set it to `1`.

The word histogram lowercases words and removes common English stop words and numeric-only tokens before counting.

```python
initial_state = {
    "page_inputs": PAGE_INPUTS,
    "messages": [],
}
final_state = app.invoke(initial_state)
```
